# Проверка API и записей в PostgreSQL

Notebook проверяет полный сценарий:

- `POST /presentations` — загрузка презентации;
- `GET /presentations` — проверка списка презентаций;
- SELECT из реляционной PostgreSQL с join, чтобы видеть название презентации рядом с чанками;
- SELECT из vector table, чтобы видеть реально добавленные RAG-документы;
- `DELETE /presentations/{presentation_id}` — опционально, если включить `CHECK_DELETE = True`.

In [ ]:
from __future__ import annotations

import json
import mimetypes
import sys
import uuid
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.project_config as settings

## 1. Настройки проверки

Заполни пути к файлам. Если хочешь проверить удаление через API, поставь `CHECK_DELETE = True`.

In [ ]:
BASE_URL = "http://127.0.0.1:8000"

PPTX_PATH = Path(r"C:\\path\\to\\presentation.pptx")
PDF_PATH = Path(r"C:\\path\\to\\presentation.pdf")

ADDITIONAL_INFO = ""
REPORT_NAME = None
PRESENTATION_ID = None

TIMEOUT_SECONDS = 900
LIST_LIMIT = 20
DB_LIMIT = 30

# Осторожно: True удалит только что загруженную презентацию через DELETE endpoint.
CHECK_DELETE = False

In [ ]:
def print_section(title: str) -> None:
    """Печатает визуальный разделитель для результата отдельного шага."""
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def print_json(value) -> None:
    """Печатает JSON или Python-объект в читаемом виде."""
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def validate_paths(pptx_path: Path, pdf_path: Path | None) -> None:
    """Проверяет наличие PPTX и PDF перед отправкой в API."""
    if not pptx_path.exists() or not pptx_path.is_file():
        raise FileNotFoundError(f"Не найден PPTX-файл: {pptx_path}")
    if pptx_path.suffix.lower() != ".pptx":
        raise ValueError(f"Файл `{pptx_path}` должен иметь расширение `.pptx`.")

    if pdf_path is None:
        return
    if not pdf_path.exists() or not pdf_path.is_file():
        raise FileNotFoundError(f"Не найден PDF-файл: {pdf_path}")
    if pdf_path.suffix.lower() != ".pdf":
        raise ValueError(f"Файл `{pdf_path}` должен иметь расширение `.pdf`.")


validate_paths(PPTX_PATH, PDF_PATH)
print("PPTX:", PPTX_PATH)
print("PDF:", PDF_PATH)
print("BASE_URL:", BASE_URL)

## 2. HTTP helpers для FastAPI

In [ ]:
def build_multipart_body(*, fields: dict[str, str], files: dict[str, Path], boundary: str) -> bytes:
    """Собирает multipart/form-data тело запроса из полей и файлов."""
    line_break = b"\r\n"
    body_parts: list[bytes] = []

    for field_name, field_value in fields.items():
        body_parts.append(f"--{boundary}".encode("utf-8"))
        body_parts.append(f'Content-Disposition: form-data; name="{field_name}"'.encode("utf-8"))
        body_parts.append(b"")
        body_parts.append(field_value.encode("utf-8"))

    for field_name, file_path in files.items():
        content_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
        body_parts.append(f"--{boundary}".encode("utf-8"))
        body_parts.append(
            (
                f'Content-Disposition: form-data; name="{field_name}"; '
                f'filename="{file_path.name}"'
            ).encode("utf-8")
        )
        body_parts.append(f"Content-Type: {content_type}".encode("utf-8"))
        body_parts.append(b"")
        body_parts.append(file_path.read_bytes())

    body_parts.append(f"--{boundary}--".encode("utf-8"))
    body_parts.append(b"")
    return line_break.join(body_parts)


def request_json(method: str, url: str, *, data: bytes | None = None, headers: dict[str, str] | None = None):
    """Выполняет HTTP-запрос и возвращает статус, текст и JSON при наличии."""
    request = Request(url=url, data=data, headers=headers or {}, method=method)
    try:
        with urlopen(request, timeout=TIMEOUT_SECONDS) as response:
            status_code = response.getcode()
            response_text = response.read().decode("utf-8", errors="replace")
    except HTTPError as error:
        status_code = error.code
        response_text = error.read().decode("utf-8", errors="replace")
    except URLError as error:
        raise RuntimeError(f"Не удалось подключиться к `{url}`: {error.reason}") from error

    try:
        response_json = json.loads(response_text)
    except json.JSONDecodeError:
        response_json = None
    return status_code, response_text, response_json


def upload_presentation() -> tuple[int, str, dict | None]:
    """Отправляет презентацию в `POST /presentations`."""
    boundary = f"----presentation-upload-{uuid.uuid4().hex}"
    fields = {"additional_info": ADDITIONAL_INFO}
    if REPORT_NAME:
        fields["report_name"] = REPORT_NAME
    if PRESENTATION_ID:
        fields["presentation_id"] = PRESENTATION_ID

    files = {"pptx_file": PPTX_PATH}
    if PDF_PATH is not None:
        files["pdf_file"] = PDF_PATH

    body = build_multipart_body(fields=fields, files=files, boundary=boundary)
    return request_json(
        "POST",
        f"{BASE_URL.rstrip('/')}/presentations",
        data=body,
        headers={
            "Content-Type": f"multipart/form-data; boundary={boundary}",
            "Content-Length": str(len(body)),
        },
    )


def list_presentations() -> tuple[int, str, dict | None]:
    """Проверяет endpoint `GET /presentations`."""
    query = urlencode({"limit": LIST_LIMIT})
    return request_json("GET", f"{BASE_URL.rstrip('/')}/presentations?{query}")


def delete_presentation(presentation_id: str) -> tuple[int, str, dict | None]:
    """Проверяет endpoint `DELETE /presentations/{presentation_id}`."""
    return request_json("DELETE", f"{BASE_URL.rstrip('/')}/presentations/{presentation_id}")

## 3. SQL helpers для реляционной и векторной PostgreSQL

In [ ]:
def quote_identifier(identifier: str) -> str:
    """Безопасно экранирует SQL-идентификатор двойными кавычками."""
    return '"' + identifier.replace('"', '""') + '"'


def quote_table_name(table_name: str, schema_name: str | None = None) -> str:
    """Возвращает SQL-имя таблицы с учётом схемы."""
    if schema_name:
        return f"{quote_identifier(schema_name)}.{quote_identifier(table_name)}"
    if "." in table_name:
        return ".".join(quote_identifier(part) for part in table_name.split("."))
    return quote_identifier(table_name)


def row_to_dict(row) -> dict:
    """Преобразует SQLAlchemy Row в обычный dict."""
    return dict(row._mapping)


def select_relational_records(presentation_id: str) -> list[dict]:
    """Читает презентацию и чанки из реляционной PostgreSQL через join с названием презентации."""
    from sqlalchemy import create_engine, text

    presentations_table = quote_table_name(settings.PRESENTATIONS_TABLE)
    chunks_table = quote_table_name(settings.CHUNKS_TABLE)
    query = text(
        f"""
        SELECT
            p.id::text AS presentation_id,
            p.report_name,
            p.link_on_file,
            c.slide_sequence_number,
            c.chunk_number,
            LEFT(c.chunk_summary, 700) AS chunk_summary_preview,
            LEFT(c.source_slide_text, 700) AS source_slide_text_preview
        FROM {presentations_table} AS p
        JOIN {chunks_table} AS c
            ON c.presentation_id = p.id
        WHERE p.id = CAST(:presentation_id AS UUID)
        ORDER BY c.slide_sequence_number, c.chunk_number
        LIMIT :limit
        """
    )
    engine = create_engine(settings.RELATIONAL_CONNECTION_STRING)
    try:
        with engine.begin() as connection:
            result = connection.execute(query, {"presentation_id": presentation_id, "limit": DB_LIMIT})
            return [row_to_dict(row) for row in result]
    finally:
        engine.dispose()


async def select_vector_records(presentation_id: str) -> list[dict]:
    """Читает RAG-документы из vector table и показывает название презентации из metadata columns."""
    from sqlalchemy import text
    from sqlalchemy.ext.asyncio import create_async_engine

    vector_table = quote_table_name(settings.VECTOR_TABLE, settings.VECTOR_SCHEMA)
    query = text(
        f"""
        SELECT
            {quote_identifier(settings.VECTOR_ID_COLUMN)} AS vector_id,
            report_name,
            presentation_id,
            type,
            slide_number,
            chunk_number,
            total_chunks,
            LEFT(content, 700) AS content_preview
        FROM {vector_table}
        WHERE presentation_id = :presentation_id
        ORDER BY type, slide_number, chunk_number
        LIMIT :limit
        """
    )
    engine = create_async_engine(settings.VECTOR_CONNECTION_STRING)
    try:
        async with engine.begin() as connection:
            result = await connection.execute(query, {"presentation_id": presentation_id, "limit": DB_LIMIT})
            return [row_to_dict(row) for row in result]
    finally:
        await engine.dispose()

## 4. Загрузка презентации

In [ ]:
print_section("POST /presentations")
upload_status, upload_text, upload_json = upload_presentation()
print("HTTP", upload_status)
print_json(upload_json if upload_json is not None else upload_text)

if upload_status >= 400:
    raise RuntimeError("Загрузка завершилась ошибкой. Проверь ответ выше и логи сервера.")

uploaded_presentation_id = upload_json["presentation_id"]
print("\nЗагруженная presentation_id:", uploaded_presentation_id)

## 5. Проверка `GET /presentations`

In [ ]:
print_section("GET /presentations")
list_status, list_text, list_json = list_presentations()
print("HTTP", list_status)
print_json(list_json if list_json is not None else list_text)

if list_status >= 400:
    raise RuntimeError("GET /presentations завершился ошибкой.")

## 6. Что реально добавилось в реляционную PostgreSQL

Здесь используется join таблицы презентаций и таблицы чанков, чтобы рядом с каждым чанком видеть `report_name`.

In [ ]:
print_section("Relational PostgreSQL: presentations JOIN chunks")
relational_rows = select_relational_records(uploaded_presentation_id)
print(f"Найдено строк: {len(relational_rows)}")
print_json(relational_rows)

if not relational_rows:
    raise RuntimeError("В реляционной БД не найдены записи по загруженной presentation_id.")

## 7. Что реально добавилось в vector table

Здесь показываются документы, которые попали в PostgreSQL/pgvector. В ответе должны быть документы типа `report` и `slide_chunk`.

In [ ]:
print_section("Vector PostgreSQL / pgvector table")
vector_rows = await select_vector_records(uploaded_presentation_id)
print(f"Найдено vector-документов: {len(vector_rows)}")
print_json(vector_rows)

if not vector_rows:
    raise RuntimeError("В vector table не найдены документы по загруженной presentation_id.")

## 8. Опциональная проверка `DELETE /presentations/{presentation_id}`

Эта ячейка удалит загруженную презентацию только если `CHECK_DELETE = True`.

In [ ]:
if CHECK_DELETE:
    print_section(f"DELETE /presentations/{uploaded_presentation_id}")
    delete_status, delete_text, delete_json = delete_presentation(uploaded_presentation_id)
    print("HTTP", delete_status)
    print_json(delete_json if delete_json is not None else delete_text)

    if delete_status >= 400:
        raise RuntimeError("DELETE /presentations/{presentation_id} завершился ошибкой.")

    print_section("Проверка БД после DELETE")
    relational_after_delete = select_relational_records(uploaded_presentation_id)
    vector_after_delete = await select_vector_records(uploaded_presentation_id)
    print("Relational rows after delete:", len(relational_after_delete))
    print("Vector rows after delete:", len(vector_after_delete))
else:
    print("CHECK_DELETE = False. Удаление пропущено, загруженные данные оставлены в БД.")